In [3]:
import torch
import torch.nn as nn
import PIL
import urllib.request
from datasets import load_dataset
from transformers import ViTForImageClassification, AutoImageProcessor, Trainer, TrainingArguments, pipeline
from transformers import CLIPProcessor, CLIPModel, Blip2Processor, Blip2ForConditionalGeneration
from google import genai

vit from the scratch

In [4]:
class PatchEmbedding(nn.Module):
    def __init__(self, input_size, embed_dim, patch_size=16):
        super().__init__()
        self.conv2d = nn.Conv2d(input_size, embed_dim, kernel_size=patch_size, stride=patch_size)
    
    def forward(self, X):
        X = self.conv2d(X)
        X = X.flatten(start_dim=2)
        return X.transpose(1, 2)
    
class Vit(nn.Module):
    def __init__(self, image_size=224, patch_size=16, in_channels=3, num_classes=1000, embed_dim=768, depth=12,
                 num_heads=12, ff_dim=3072, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(in_channels, embed_dim, patch_size)
        cls_init = torch.randn(1, 1, embed_dim) * 0.02
        self.cls_token = nn.Parameter(cls_init)
        num_patches = (image_size // patch_size)**2
        pos_init = torch.randn(1, num_patches+1, embed_dim) * 0.02
        self.pos_embed = nn.Parameter(pos_init)
        self.dropout = nn.Dropout(p=dropout)
        encoder_layer = nn.TransformerEncoderLayer(embed_dim, num_heads, ff_dim, dropout, 'gelu', batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, depth)
        self.layer_norm = nn.LayerNorm(embed_dim)
        self.output = nn.Linear(embed_dim, num_classes)
        
    def forward(self, X):
        Z = self.patch_embed(X)
        cls_expd = self.cls_token.expand(Z.shape[0], -1, -1)
        Z = torch.cat((cls_expd, Z), dim=1)
        Z = Z + self.pos_embed
        Z = self.dropout(Z)
        Z = self.encoder(Z)
        Z = self.layer_norm(Z[:, 0])
        logits = self.output(Z)
        return logits

In [5]:
my_vit_model = Vit()
batch = torch.randn(4, 3, 224, 224)
logits = my_vit_model(batch)
logits

tensor([[ 0.0120, -0.2425, -0.2022,  ...,  1.0835, -0.3870, -0.8560],
        [-0.0773,  0.0980,  0.0645,  ...,  0.7481, -0.1539, -1.5311],
        [-0.3623,  0.2725, -0.3483,  ...,  0.3572, -0.4057, -1.0400],
        [-0.4818,  0.2098,  0.0794,  ...,  0.9017, -0.4438, -1.0487]],
       grad_fn=<AddmmBackward0>)

fine-tunning pre-trained vit

In [7]:
pets = load_dataset('timm/oxford-iiit-pet')

Generating test split: 100%|██████████| 3669/3669 [00:01<00:00, 2930.48 examples/s]


In [8]:
model_dir = "google/vit-base-patch16-224-in21k"
vit_model = ViTForImageClassification.from_pretrained(model_dir, num_labels=37)
vit_processor = AutoImageProcessor.from_pretrained(model_dir, use_fast=True)

Loading weights: 100%|██████████| 6/6 [00:00<00:00, 12122.27it/s]
[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
encoder.layer.{0...11}.attention.output.dense.bias      | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.value.weight | UNEXPECTED | 
encoder.layer.{0...11}.output.dense.weight              | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.value.bias   | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_before.bias            | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.query.bias   | UNEXPECTED | 
pooler.dense.weight                                     | UNEXPECTED | 
encoder.layer.{0...11}.layernorm_after.weight           | UNEXPECTED | 
encoder.layer.{0...11}.intermediate.dense.weight        | UNEXPECTED | 
encoder.layer.{0...11}.attention.output.dense.wei

In [9]:
def vit_collate_fn(batch):
    images = [example['image'] for example in batch]
    labels = [example['label'] for example in batch]
    inputs = vit_processor(images, return_tensors='pt', do_convert_rgb=True)
    inputs['labels'] = torch.tensor(labels)
    return inputs

In [ ]:
args = TrainingArguments('pets', per_device_train_batch_size=16, eval_strategy='epoch', num_train_epochs=3, remove_unused_columns=False)
trainer = Trainer(vit_model, args, vit_collate_fn, pets['train'], pets['test'])
train_ouput = trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,3.568009
2,No log,3.471489
3,3.539045,3.377263


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.37it/s]


## CLIP

In [2]:
clip_dir = "openai/clip-vit-base-patch32"
clip_pipeline = pipeline(task='zero-shot-image-classification', model_id=clip_dir, device_map='auto', dtype='auto')
candidate_labels = ['cricket', 'ladybug', 'spider']
image_url = 'https://homl.info/ladybug'
results = clip_pipeline(image=image_url, candidate_labels=candidate_labels, hypothesis_template='This is a photo of a {}.')

[transformers] No model was supplied, defaulted to openai/clip-vit-base-patch32 and revision 3d74acf.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 5895.02it/s]


In [3]:
print(results)

[{'score': 0.9972797632217407, 'label': 'ladybug'}, {'score': 0.001655837520956993, 'label': 'spider'}, {'score': 0.0010643504792824388, 'label': 'cricket'}]


In [5]:
clip_processor = CLIPProcessor.from_pretrained(clip_dir)
clip_model = CLIPModel.from_pretrained(clip_dir)
image = PIL.Image.open(urllib.request.urlopen(image_url)).convert('RGB')
captions = [f'This is a photo of a {label}.' for label in candidate_labels]
inputs = clip_processor(images=[image], text=captions, return_tensors='pt', padding=True)
with torch.no_grad():
    outputs = clip_model(**inputs)
    
text_features  = outputs.text_embeds
image_features = outputs.image_embeds

similarities = image_features @ text_features.T
print(similarities)

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 40890.97it/s]


tensor([[0.2336, 0.3021, 0.2380]])


In [6]:
temperature = clip_model.logit_scale.detach().exp()
rescaled_similarities = similarities * temperature
probas = torch.nn.functional.softmax(rescaled_similarities, dim=1)
print(probas)

tensor([[0.0011, 0.9973, 0.0017]])


### BLIP

In [ ]:
blip_dir = 'Salesforce/blip2-opt-2.7b'
blip_processor = Blip2Processor.from_pretrained(blip_dir)
blip_model = Blip2ForConditionalGeneration.from_pretrained(blip_dir, device_map='auto', dtype=torch.float16)

image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = PIL.Image.open(urllib.request.urlopen(image_url))
inputs = blip_processor(image, return_tensors='pt')
inputs = inputs.to(blip_model.device, dtype=torch.float16)
with torch.no_grad():
    outputs = blip_model.generate(**inputs)
    
text = blip_processor.batch_decode(outputs)
print(text)

Loading weights: 100%|██████████| 1247/1247 [00:12<00:00, 101.47it/s]
/home/damian/New Folder/.venv/lib/python3.14/site-packages/accelerate/utils/modeling.py:1615: UserWarning: The following device_map keys do not match any submodules in the model: ['query_tokens']
  warnings.warn(
Some parameters are on the meta device because they were offloaded to the cpu.
/home/damian/New Folder/.venv/lib/python3.14/site-packages/transformers/generation/utils.py:1638: UserWarning: Using the model-agnostic default `max_length` (=53) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


['<image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image></s>two cats laying on a couch\n']


## Gemini

In [ ]:
gemini_key = "Your token" #erase later
gemini_client = genai.Client(api_key=gemini_key)
cats_photo = gemini_client.files.upload(file="kitties.png")
question = 'What animal and how many? Format: [animal, number]'
response = gemini_client.models.generate_content(model='gemini-3.6-flash', contents=[cats_photo, question])
print(response.text)

[cat, 2]
